# MANIQA Debug Forward Walkthrough

이 노트북은 `debug_forward.py`와 같은 위치에 있으며, 한 장의 이미지 또는 random tensor가 MANIQA를 통과하는 전체 흐름을 순서대로 확인하기 위한 학습용 노트북입니다.

목표는 성능 재현이 아니라 논문 Figure 2의 구조를 코드 기준으로 따라가는 것입니다.

## 1. 환경 확인

VSCode에서 이 노트북의 kernel은 반드시 `MANIQA/.venv`를 선택하세요.

현재 프로젝트에서 확인된 디버그용 패키지 조합은 Python 3.12.10, 최신 PyTorch 계열입니다. 논문 재현 환경인 Python 3.7.9 + PyTorch 1.8.0과는 다르지만, forward shape 분석은 가능합니다.

In [ ]:
import sys
import torch

print(sys.executable)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())

## 2. 디버그 함수 import

`debug_forward.py`에는 다음 역할이 들어 있습니다.

- 입력 이미지 로드 또는 random tensor 생성
- 224x224 resize와 normalize
- MANIQA 모델 생성
- checkpoint가 있으면 로드
- `model(x, debug=True)`로 단계별 shape 출력

In [ ]:
from debug_forward import load_image_tensor, load_checkpoint_if_requested, setup_seed
from models.maniqa import MANIQA

## 3. 입력 이미지 준비

논문 Figure 2의 첫 단계는 입력 이미지입니다.

여기서는 repo에 포함된 `image/kunkun.png`를 사용합니다. 실제 파일이 없거나 shape만 보고 싶으면 다음 코드에서 `use_random=True`로 바꾸면 됩니다.

In [ ]:
setup_seed(20)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
x = load_image_tensor('image/kunkun.png', img_size=224, use_random=False).to(device)
print('input ready:', tuple(x.shape), x.device)

## 4. 모델 생성

논문 Figure 2의 전체 흐름은 다음 순서입니다.

1. 입력 이미지
2. 224x224 resize 또는 crop
3. ViT patch embedding
4. ViT 여러 layer feature 추출
5. feature concatenate
6. TAB: channel dimension attention
7. SSTB: window attention, shifted window attention
8. Dual Branch: score branch, weight branch
9. patch score와 weight로 최종 quality score 계산

`vit_pretrained=False`로 두면 pretrained weight 다운로드 없이 구조 확인이 가능합니다.

In [ ]:
model = MANIQA(
    embed_dim=768,
    num_outputs=1,
    dim_mlp=768,
    patch_size=8,
    img_size=224,
    window_size=4,
    depths=[2, 2],
    num_heads=[4, 4],
    num_tab=2,
    scale=0.8,
    vit_pretrained=False,
).to(device)

model.eval()
print('model ready')

## 5. Optional checkpoint 로드

checkpoint가 없으면 이 단계는 건너뛰어도 됩니다.

지금 목적은 성능이 아니라 tensor shape과 forward 흐름 확인이므로, checkpoint 없이도 괜찮습니다.

In [ ]:
# checkpoint_path = 'ckpt_koniq10k.pt'
checkpoint_path = ''
load_checkpoint_if_requested(model, checkpoint_path, device)

## 6. Forward 실행과 shape 출력

`debug=True`를 주면 `models/maniqa.py` 내부에서 Figure 2 단계별 shape을 출력합니다.

특히 확인할 포인트는 다음입니다.

- ViT patch embedding: `(B, 784, 768)`
- 선택된 ViT layer 6~9 feature: 각각 `(B, 784, 768)`
- concatenate 후: `(B, 784, 3072)`
- TAB channel attention weight: stage1 `(B, 3072, 3072)`, stage2 `(B, 768, 768)`
- SSTB 후 spatial feature: `(B, C, 28, 28)`
- Dual Branch 출력: patch별 score와 weight가 `(784, 1)`

In [ ]:
with torch.no_grad():
    score = model(x, debug=True)

print('final predicted score:', score.detach().cpu().numpy())

## 7. Random tensor로 다시 확인

이미지 파일, OpenCV, checkpoint 문제가 생겨도 모델 구조만 확인하고 싶으면 random tensor를 사용하면 됩니다.

In [ ]:
x_random = load_image_tensor('', img_size=224, use_random=True).to(device)

with torch.no_grad():
    random_score = model(x_random, debug=True)

print('random final predicted score:', random_score.detach().cpu().numpy())